# Session 7: Text Classification Pipelines with scikit-learn

## Overview

Session 6 built a working spam filter by hand. Here you turn it into one you could ship. You will wrap every step
in a scikit-learn **pipeline**, replace raw counts with **TF-IDF** features, compare naive Bayes with two stronger
linear classifiers, tune the whole pipeline with a **grid search**, choose a decision threshold for the precision
the business needs, and **save** the finished filter to a file that another program can load.

By the end, you will be able to:

- Build text-classification pipelines that go from raw strings to predictions.
- Explain TF-IDF and character n-grams, and when they help.
- Compare classifiers with several metrics at once, and explain why precision and recall depend on the threshold
  while average precision does not.
- Tune vectoriser and model settings together with `GridSearchCV`.
- Choose a threshold from cross-validated scores to meet a precision target.
- Save a model and its threshold with `joblib`, reload it, and classify new messages.

**Prerequisites:** Session 6 (naive Bayes, the document-term matrix); Session 5 (precision, recall, thresholds).

**Dataset:** [UCI SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection), exactly as
in Session 6: 5,171 unique SMS messages labelled ham or spam, with the same train/test split.

## Where this fits

| Part | Sessions | Task | Algorithms |
| --- | --- | --- | --- |
| 1. Regression | 1, 2, 3, 4 | Predict a number | Linear regression by gradient descent and in scikit-learn, tree ensembles, explaining black-box models |
| 2. Classification | 5, 6, 7, 8, 9 | Predict a category | Logistic regression, naive Bayes, linear text classifiers, comparing classifiers **← you are here** |
| 3. Clustering | 10, 11 | Find groups without labels | k-means, hierarchical clustering |
| 4. Topic modelling | 12 | Find themes in text | LDA, NMF |

**This session's question:** Can we build a spam filter that almost never blocks a real message, and package it
for deployment?

## Step 1: Set up the environment

This module has one pinned `requirements.txt` (in the module folder) for all twelve sessions. If you have
not done it yet, create a virtual environment and install it once, from the module folder:

```bash
python3.11 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\Activate.ps1
pip install -r requirements.txt
jupyter notebook
```

This session uses:

| Package | Why we need it |
| --- | --- |
| scikit-learn | Vectorisers, classifiers, pipelines, grid search, metrics |
| joblib | Saving and loading the finished pipeline |
| pandas | Tables |
| numpy | Arrays |
| matplotlib | Charts |

Run the cell below. It imports what this notebook needs and prints the versions. The numbers quoted in
this notebook come from these exact versions; a different version can shift the last decimal place.

In [ ]:
import io
import urllib.request
import zipfile

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict, cross_validate, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.svm import LinearSVC

from importlib.metadata import version

for package in ["scikit-learn", "joblib", "pandas", "numpy", "matplotlib"]:
    print(f"{package:13s} {version(package)}")

## Step 2: Load the messages with Session 6's split

Same download, same duplicate removal, same `random_state`, so the test messages are the ones Session 6 was
tested on. The target is now 1 for spam and 0 for ham, which is what scikit-learn's metrics expect for the
"positive" class.

In [ ]:
URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
with zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL).read())) as archive:
    lines = archive.read("SMSSpamCollection").decode("utf-8").splitlines()
sms = pd.DataFrame([line.split("\t", 1) for line in lines if line.strip()], columns=["label", "message"])
sms = sms.drop_duplicates("message").reset_index(drop=True)
sms["spam"] = (sms["label"] == "spam").astype(int)

train, test = train_test_split(sms, test_size=0.2, stratify=sms["label"], random_state=42)
X_train, y_train = train["message"], train["spam"]
X_test, y_test = test["message"], test["spam"]
print(f"train: {len(X_train):,} messages ({y_train.sum()} spam), test: {len(X_test):,} ({y_test.sum()} spam)")

## Step 3: Session 6's model as a pipeline, and four metrics

A pipeline takes **raw strings** in and gives predictions out. Because the vectoriser is inside it,
cross-validation re-builds the vocabulary on each fold's training messages only, so no word from a validation
fold leaks into training.

`cross_validate` can score several metrics in one go. For a spam filter:

- **precision:** of the messages we block, the share that really are spam. A low value means real messages lost.
- **recall:** of the spam, the share we block.
- **F1:** a single number that balances the two (their harmonic mean).
- **average precision (AP):** the area under the precision-recall curve. It looks at the model's *scores*, not
  its yes/no decisions, so it measures how well the model **ranks** spam above ham, whatever threshold is used
  later.

In [ ]:
TOKENS = r"[a-z0-9£$']+"          # Session 6's tokeniser
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
METRICS = ["precision", "recall", "f1", "average_precision"]


def evaluate(pipeline):
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=METRICS)
    return {metric: scores[f"test_{metric}"].mean() for metric in METRICS}


results = {}
results["counts + naive Bayes"] = evaluate(make_pipeline(CountVectorizer(token_pattern=TOKENS), MultinomialNB()))
print(pd.DataFrame(results).T.round(3))

**Read the output:** cross-validated precision 0.972 and recall 0.906, close to Session 6's test results. This is
the baseline to beat.

## Step 4: TF-IDF features

Raw counts treat every word alike. But a word that appears in almost every message ("you", "the") says little,
while a word that appears in few messages ("prize") is distinctive. **TF-IDF** (term frequency × inverse document
frequency) re-weights each count:

$$\text{tf-idf}(w, d) = \underbrace{\text{count of } w \text{ in } d}_{\text{tf}} \times \underbrace{\left(\log\frac{1 + N}{1 + \text{df}(w)} + 1\right)}_{\text{idf}}$$

where N is the number of training messages and df(w) the number of them that contain w. Words found everywhere
get an idf near 1; rare words get a large one. Each message's vector is then scaled to length 1, so that long
messages do not get larger scores just for being long.

In [ ]:
tfidf = TfidfVectorizer(token_pattern=TOKENS).fit(X_train)
example = "URGENT! You have won a prize. Call now to claim it"
weights = pd.Series(tfidf.transform([example]).toarray()[0], index=tfidf.get_feature_names_out())
print(weights[weights > 0].sort_values(ascending=False).round(3).to_string())

**Read the output:** in this message, the distinctive words "urgent", "won", "prize" and "claim" get the highest
weights (about 0.4), while common words such as "a", "you" and "to" get the lowest (below 0.2). (The single letter "a" is kept by our tokeniser;
the default scikit-learn tokeniser drops one-character words.)

## Step 5: Compare classifiers on TF-IDF features

Three classifiers that all work well on sparse text features:

- **Multinomial naive Bayes** (Session 6).
- **Logistic regression** (Session 5), with its default regularisation `C=1`.
- **Linear support vector machine** (`LinearSVC`): also a linear model, but instead of the log-loss it maximises
  the *margin*, the gap between the decision boundary and the closest messages of each class. It outputs a score,
  not a probability.

In [ ]:
for name, classifier in [("TF-IDF + naive Bayes", MultinomialNB()),
                         ("TF-IDF + logistic regression", LogisticRegression(max_iter=1000)),
                         ("TF-IDF + linear SVM", LinearSVC())]:
    results[name] = evaluate(make_pipeline(TfidfVectorizer(token_pattern=TOKENS), classifier))
print(pd.DataFrame(results).T.round(3))

**Read the output:** something strange happens with naive Bayes. On TF-IDF features its **recall collapses** to
about 0.6, while its precision is perfect and its **average precision barely changes** (0.946 against 0.958).

The explanation is in the metrics. AP measures *ranking*, and the ranking is still good. Precision and recall
measure *decisions at a threshold*, and the threshold is now wrong: TF-IDF values are small fractions, so the
evidence from the words is much weaker than with counts, and the prior (87% ham) wins for many spam messages.
The model is not worse at telling spam from ham; its default cut-off is in the wrong place.

Logistic regression shows the same pattern, less strongly (recall 0.751, AP 0.970): its default penalty `C=1` is
too strong for these small TF-IDF values. The linear SVM has the best AP (0.979) and F1 (0.950).

**Lesson: compare models on a threshold-free metric such as AP, then choose the threshold separately** (Step 7).

## Step 6: Tune the whole pipeline with a grid search

Settings of the vectoriser and the classifier interact, so we tune them together. `GridSearchCV` tries every
combination with cross-validation and keeps the best. Pipeline step settings are named `step__parameter`.

We include **character n-grams** (`analyzer="char_wb"`): instead of words, the features are all sequences of 2 to
5 characters inside words ("pri", "priz", "rize"...). They cope with spelling tricks, missing spaces and text-speak
("FR33", "STOP2stop", "txt"), all common in spam. `sublinear_tf=True` uses 1 + log(count) instead of the raw count,
so a word repeated five times does not count five times as much.

In [ ]:
pipeline = Pipeline([("features", TfidfVectorizer(sublinear_tf=True)), ("model", LinearSVC())])
grid = [
    {"features__analyzer": ["word"], "features__token_pattern": [TOKENS],
     "features__ngram_range": [(1, 1), (1, 2)], "model__C": [0.1, 1, 10]},
    {"features__analyzer": ["char_wb"], "features__ngram_range": [(2, 5)], "model__C": [0.1, 1, 10]},
]
search = GridSearchCV(pipeline, grid, cv=cv, scoring="average_precision", n_jobs=-1).fit(X_train, y_train)

table = pd.DataFrame(search.cv_results_["params"])
table["CV average precision"] = search.cv_results_["mean_test_score"]
print(table.drop(columns="features__token_pattern").sort_values("CV average precision", ascending=False)
      .round(4).to_string(index=False))
print("\nbest:", {k: v for k, v in search.best_params_.items() if k != "features__token_pattern"})

**Read the output:** the three character n-gram pipelines top the table, with an average precision of up to
0.986, ahead of the best word-based pipeline (word pairs, 0.981). Within each kind of feature, C makes a
difference of a few thousandths; the best is `C=1`. `GridSearchCV` has refitted the best pipeline on all the training messages; it is
`search.best_estimator_`.

Nine settings × 5 folds = 45 fits, in a few seconds. With more settings, use `RandomizedSearchCV`, which tries a
random sample of combinations.

## Step 7: Choose the threshold for 99% precision

The operator's requirement: **at most 1 in 100 blocked messages may be legitimate** (precision ≥ 0.99). Find the
threshold on the SVM's score that meets it with the highest recall.

We need scores the model did not train on, without touching the test set: `cross_val_predict` gives every
training message a score from the fold in which it was held out.

In [ ]:
best = search.best_estimator_
cv_scores = cross_val_predict(best, X_train, y_train, cv=cv, method="decision_function")
precision, recall, thresholds = precision_recall_curve(y_train, cv_scores)

meets_target = np.flatnonzero(precision[:-1] >= 0.99)        # the last point has no threshold
chosen = meets_target[np.argmax(recall[meets_target])]          # ... of those, the one with the best recall
threshold = thresholds[chosen]
default = np.searchsorted(thresholds, 0)
print(f"chosen threshold {threshold:6.3f}: CV precision {precision[chosen]:.3f}, recall {recall[chosen]:.3f}")
print(f"default threshold {0:6.3f}: CV precision {precision[default]:.3f}, recall {recall[default]:.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(recall, precision)
ax.axhline(0.99, color="grey", linestyle="--", label="precision target 0.99")
ax.set_xlabel("recall (share of spam blocked)")
ax.set_ylabel("precision (share of blocked that is spam)")
ax.set_ylim(0.5, 1.01)
ax.set_title("Precision-recall curve (cross-validated)")
ax.legend()
plt.show()

**Read the output:** the curve shows the trade-off: to the left, a strict threshold blocks only obvious spam with
near-perfect precision; moving right, recall rises and precision eventually falls. The default threshold (0) already gives
a cross-validated precision of 0.994, with recall 0.920. Because the target is 0.99, not 1.0, we can afford to
lower the threshold to -0.164: precision stays at or above the target, and recall rises to 0.948.

The threshold is part of the model. It must be saved with it (Step 10).

## Step 8: Final test

Score the 1,035 test messages once, with the chosen pipeline and threshold, and compare with Session 6.

In [ ]:
test_scores = best.decision_function(X_test)
predicted = (test_scores >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, predicted).ravel()
print(f"ham wrongly blocked: {fp} of {tn + fp}    spam missed: {fn} of {fn + tp}")
print(classification_report(y_test, predicted, target_names=["ham", "spam"], digits=3))

**Read the output:** compare with Session 6's naive Bayes on the same messages (4 ham blocked, 11 spam missed,
precision 0.968, recall 0.916). The tuned pipeline blocks **2** legitimate messages and misses only **4** spam:
precision 0.984, recall 0.969. Both kinds of error are roughly halved or better.

Precision on the test set (0.984) is a little below the 0.99 target. With 131 spam messages in the test set, one
false alarm more or less moves precision by almost a percentage point, so this is within the noise of a small
test set. Before promising 0.99 to the operator, you would check it on more data.

`classification_report` shows precision, recall and F1 for **both** classes. *Support* is the number of test
messages in each class.

## Step 9: What does it look for?

A linear model's weights can be read like naive Bayes's log ratios. With character n-grams, the features are
fragments of words.

In [ ]:
weights = pd.Series(best["model"].coef_[0], index=best["features"].get_feature_names_out())
print("strongest spam fragments:", weights.nlargest(15).index.tolist())
print("strongest ham fragments: ", weights.nsmallest(15).index.tolist())

**Read the output:** almost all of the strongest spam fragments are **digits and currency**: the starts of phone
numbers (" 0", "08", "09", " 08"), prices and amounts ("50", "00", " £", "15"), plus "xt" (from "txt") and "mob" (from
"mobile"). Spam asks you to call or text a number, or offers money. The ham fragments are the language of personal
messages: "i", "me", "my", ":)", "..". Spaces in the fragments mark the start or end of a word (`char_wb` pads each word with a space).
Character features are less readable than words, the price of their robustness.

## Step 10: Save the filter, load it, use it

A model in a notebook helps nobody. `joblib` saves the fitted pipeline (vocabulary, IDF weights and classifier
weights) to a single file. We save the threshold with it, so that whoever loads the model applies the same
decision rule.

In [ ]:
joblib.dump({"pipeline": best, "threshold": float(threshold)}, "sms_spam_filter.joblib")

loaded = joblib.load("sms_spam_filter.joblib")


def is_spam(messages, model=loaded):
    return model["pipeline"].decision_function(messages) >= model["threshold"]


new_messages = [
    "Congratulations! You've been selected for a FREE holiday. Txt GO to 80082 to claim. T&Cs apply",
    "Are we still on for dinner tonight? I'll be there by 8",
    "Your parcel is waiting. Confirm delivery at www.track-parcel-now.com within 24hrs",
    "Ur awarded a £500 gift card! Call 09061234567 now",
]
for message, flag in zip(new_messages, is_spam(new_messages)):
    print(f"{'SPAM' if flag else 'ham ':4s}  {message}")

**Read the output:** the reloaded filter handles brand-new messages: the holiday, parcel and gift-card messages
are flagged as spam and the dinner message is let through. The parcel-delivery scam is a style of spam that became
common years after this dataset was collected; the filter catches this one from its web address and urgency, but
new styles of spam will keep appearing that it has never seen. A deployed filter must be **monitored and retrained** as
spam changes, which is what the [MLOps Skilling Course](../../11.%20MLOps%20Skilling%20Course) is about.

Two practical rules for saved models: load them only from sources you trust (loading a `joblib` file can run
code), and load them with the same scikit-learn version that saved them.

## What this session hands to the next

- **Pipelines** from raw text to prediction; TF-IDF and character n-grams.
- **Model comparison on AP** (ranking), then a **threshold** chosen for the business requirement.
- **A saved, reloadable model** that carries its own decision rule.
- Session 8 extends text classification from two classes to **twenty**, and to a hierarchy of classes.

## Common mistakes and fixes

| Mistake | What goes wrong | Fix |
| --- | --- | --- |
| Fitting the vectoriser on all the data before cross-validation | Vocabulary and IDF leak from validation folds | Put the vectoriser inside the pipeline |
| Comparing models by precision or recall at the default threshold | A good ranker looks bad (TF-IDF + naive Bayes) | Compare on AP or ROC AUC; choose the threshold afterwards |
| Choosing the threshold on the test set | Optimistic results | Use `cross_val_predict` scores on the training set |
| Saving the model without its threshold | Others apply the default threshold | Save both together |
| Calling `predict_proba` on `LinearSVC` | `AttributeError` | Use `decision_function`, or wrap it in `CalibratedClassifierCV` |
| Loading a model with a different scikit-learn version | Warnings, errors or silently wrong results | Pin versions (`requirements.txt`) |

## Exercises

- [ ] Add `MultinomialNB` with an `alpha` grid of 0.01, 0.1 and 1 to the grid search (as a third dictionary, with
  `"model": [MultinomialNB()]`). Where does it rank?
- [ ] Tighten the target to precision ≥ 0.999. How much recall do you lose?
- [ ] Combine word and character features with `sklearn.pipeline.FeatureUnion`. Does it beat either alone?
- [ ] Write five spam messages of your own that try to fool the filter. Which tricks work?

Next, Session 8 classifies newsgroup posts into 20 topics: multi-class classification, a confusion matrix with 400
cells, and a two-level hierarchy of classes.